# 🚀 Imports

In [1]:
# Maths
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# String
!pip -q install tldextract
import urllib              # Split URL by structure
import tldextract          # Splid domain by structure
import re                  # Find patterns in string

def print_safe(url):
  '''Prints safe, unclickable URL.'''
  print(url.replace("://", "://\u200b"))


In [2]:
# Number of CPU cores: 2
# !lscpu

---

# ❗ IMPORTANT: SAFETY GUIDELINES AGAINST MALWARE ⚠️

> ***READ BEFORE RUNNING ANY CELL***
>
> **The `url` column is dangerous if click link**, be careful. Other columns from are safe.
> - ⚠️ Display `df` — Always use `display(df)` so that URL are  not clickable.
> - ⚠️ Use `print_safe` — Use `print_safe(url)` so that URL are  not clickable.
> - ⚠️ String Parsing — Safe but make sure not to accidentally click link
>   - `urlparse`— Splits URL by structure
>   - `tldextract`— Splits domain specifically
>   - `re` — Finds patterns in string
> - ❌ DNS/WHOIS lookups — use `safe_dns_lookup()` only, one person runs this
> - ❌ `requests.get(url)` — never, under any circumstance


In [3]:
url = "http://mail.paypa1-secure.evil.com/login?user=1%20x"

# urlparse
from urllib.parse import urlparse
p = urlparse(url)
p.scheme    # 'http'
p.netloc    # 'mail.paypa1-secure.evil.com'
p.path      # '/login'
p.query     # 'user=1%20x'

# tldextract
ext = tldextract.extract(url)
ext.subdomain  # 'mail.paypa1-secure'
ext.domain     # 'evil'
ext.suffix     # 'com'

# re
re.search(r'\d+\.\d+\.\d+\.\d+', url)  # IP address present?
re.findall(r'\d', url)                 # digits in URL
re.search(r'%[0-9a-fA-F]{2}', url)     # hex encoding present?

print_safe(url)

http://​mail.paypa1-secure.evil.com/login?user=1%20x


---

# 🧪 Import Data

Imports cleaned dataset, so can immediately start working on the ML.

In [4]:
# Link to CSV
file_path = "../data/cleaned-data.csv"
df = pd.read_csv(file_path)
df.head()

,url_len,dom_len,tld_len,is_ip,subdom_cnt,letter_cnt,digit_cnt,special_cnt,eq_cnt,qm_cnt,...,under_cnt,letter_ratio,digit_ratio,spec_ratio,is_https,slash_cnt,entropy,path_len,query_len,label
0,24,11,6,0,1,17,0,7,0,0,...,0,0.708333,0.0,0.291667,1,3,3.709148,1,0,0
1,26,14,6,0,1,19,0,7,0,0,...,0,0.730769,0.0,0.269231,0,3,3.738149,1,0,0
2,23,10,6,0,1,16,0,7,0,0,...,0,0.695652,0.0,0.304348,1,3,3.609668,1,0,0
3,19,11,6,0,0,13,0,6,0,0,...,0,0.684211,0.0,0.315789,0,3,3.576618,1,0,0
4,22,10,6,0,1,15,0,7,0,0,...,0,0.681818,0.0,0.318182,0,3,3.503998,1,0,0


---

# ML Models to Use
1. Logistic Regression
2. Random forest (most likely candidate)
3. XGBoost/LightBGM/Catboost
4. Support Vector Machine (SVM)
5. Neural networks


---

# 📊 Machine Learning 1 (Imbalance)

- Find out the best model (recall) of 22 features using the imbalance fixing method below:
  - **Weighted class** (most likely)
  - **Balanced Random Forest**
  - **SMOTE** (or others).
  - **ADASYN**


## Pre-Req: 5 ML-Helper Function

Speed up ML training using Intel CPUs.

In [7]:
from sklearnex import patch_sklearn
patch_sklearn()

Extension for Scikit-learn* enabled (https://github.com/uxlfoundation/scikit-learn-intelex)


Testing 5 ML-Helper function:

In [8]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from xgboost import XGBClassifier
import lightgbm as lgb
import warnings
warnings.filterwarnings('ignore')

def benchmark_models(X_train, X_test, y_train, y_test, use_class_weights=False, random_state=42):
    """
    Benchmarks 5 ML models that is:
        1. Logistic Regression
        2. Random forest (most likely candidate)
        3. XGBoost/LightBGM/Catboost
        4. Support Vector Machine (SVM)
        5. Neural networks

    Returns:
        Sorted DataFrame by recall (phishing=1)
    """
    # Class weights
    cw = 'balanced' if use_class_weights else None
    cw_XGBoost = (y_train==0).sum() / (y_train==1).sum() if use_class_weights else 1
    
    # Train models
    models = {
        'Logistic Regression': LogisticRegression(
            class_weight=cw, max_iter=1000, random_state=random_state
        ),
        'Random Forest': RandomForestClassifier(
            class_weight=cw, n_estimators=100, n_jobs=-1, random_state=random_state
        ),
        'XGBoost': XGBClassifier(
            scale_pos_weight=cw_XGBoost, n_jobs=-1, random_state=random_state
        ),
        'LightGBM': lgb.LGBMClassifier(
            class_weight=cw, feature_pre_filter=False, n_jobs=-1, random_state=random_state, verbose=-1
        ),
        'SVM': SVC(
            class_weight=cw, kernel='rbf', random_state=random_state
        ),
        'Neural Network': MLPClassifier(
            hidden_layer_sizes=(128, 64), max_iter=300, random_state=random_state
        )
    }
    
    # Evaluatee model using test data
    results = []
    for name, model in models.items():
        print(f'Training {name}...')
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        
        results.append({
            'Model': name,
            'Accuracy':  round(accuracy_score(y_test, y_pred), 4),
            'Precision': round(precision_score(y_test, y_pred, pos_label=1), 4),
            'Recall':    round(recall_score(y_test, y_pred, pos_label=1), 4),
            'F1 Score':  round(f1_score(y_test, y_pred, pos_label=1), 4),
        })
        print(f"  Done. Recall={results[-1]['Recall']}, Features={model.n_features_in_}")

    # Store final reults in DF
    df_ML = pd.DataFrame(results).sort_values('Recall', ascending=False).reset_index(drop=True)
    return df_ML
    

## 0. Splitting Sample Dataset

Rule-of-thumb for splitting:

| Dataset Size | Recommended Split | Rationale |
|--------------|-------------------|------------|
| < 1,000 | 60/40 or use CV | Too few samples; need more for training, CV preferred |
| 1,000 – 10,000 | 70/30 | Moderate data; larger test set improves metric reliability |
| 10,000 – 100,000 | 80/20 | Sweet spot; test set still large enough for stable evaluation |
| > 100,000 | 80/20 or 90/10 | Test set is large enough even at 10%; maximize training data |
| > 1,000,000 | 95/5 or 99/1 | Even 1% gives tens of thousands of test samples |

In [9]:
from sklearn.model_selection import train_test_split

# Split data into features and labels
X = df.drop('label', axis=1)
y = df['label']
print(f'Using {X.shape[1]} features for training.')

# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f'Using {X_train.shape[0]} (80%) samples for training, {X_test.shape[0]} (20%) samples for testing.')

Using 22 features for training.
Using 92184 (80%) samples for training, 23047 (20%) samples for testing.


## 1. Weighted Class

Balances by adding weights to the clases, minority class gets higher weight while majority is less.

In [9]:
df_weighted_class = benchmark_models(X_train, X_test, y_train, y_test, use_class_weights=True)
print('-' * 20)
print(df_weighted_class.to_string(index=False))

Training Logistic Regression...
  Done. Recall=0.9198, Features=22
Training Random Forest...
  Done. Recall=0.9069, Features=22
Training XGBoost...
  Done. Recall=0.9494, Features=22
Training LightGBM...
  Done. Recall=0.953, Features=22
Training SVM...
  Done. Recall=0.921, Features=22
Training Neural Network...
  Done. Recall=0.9277, Features=22
--------------------
              Model  Accuracy  Precision  Recall  F1 Score
           LightGBM    0.9726     0.8694  0.9530    0.9093
            XGBoost    0.9783     0.9044  0.9494    0.9263
     Neural Network    0.9814     0.9424  0.9277    0.9350
                SVM    0.9468     0.7604  0.9210    0.8330
Logistic Regression    0.9282     0.6872  0.9198    0.7867
      Random Forest    0.9797     0.9498  0.9069    0.9278


## 2. SMOTE 

Synthetic Minority Over-Sampling creates synthetic minority data that balances the dataset. It does this by interpolating between existing minority samples.

https://imbalanced-learn.org/stable/references/generated/imblearn.over_sampling.SMOTE.html

The full explanation:

https://imbalanced-learn.org/stable/introduction.html

In [7]:
from imblearn.over_sampling import SMOTE

# Initialize SMOTE
smote = SMOTE(random_state=42)

# Apply SMOTE to the training data
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

print(f"Original training set shape: {X_train.shape}")
print(f"Distribution before SMOTE: {y_train.value_counts()}")
print('-' * 20)
print(f"SMOTE training set shape: {X_train_smote.shape}")
print(f"Distribution after SMOTE: {y_train_smote.value_counts()}")

Original training set shape: (92184, 22)
Distribution before SMOTE: label
0    78912
1    13272
Name: count, dtype: int64
--------------------
SMOTE training set shape: (157824, 22)
Distribution after SMOTE: label
0    78912
1    78912
Name: count, dtype: int64


In [8]:
df_SMOTE_class = benchmark_models(X_train_smote, X_test, y_train_smote, y_test, use_class_weights=False)
print('-' * 20)
print(df_SMOTE_class.to_string(index=False))

Training Logistic Regression...
  Done. Recall=0.8963, Features=22
Training Random Forest...
  Done. Recall=0.9328, Features=22
Training XGBoost...
  Done. Recall=0.9433, Features=22
Training LightGBM...
  Done. Recall=0.943, Features=22
Training SVM...
  Done. Recall=0.9027, Features=22
Training Neural Network...
  Done. Recall=0.9521, Features=22
--------------------
              Model  Accuracy  Precision  Recall  F1 Score
     Neural Network    0.9776     0.8985  0.9521    0.9245
            XGBoost    0.9790     0.9136  0.9433    0.9282
           LightGBM    0.9725     0.8757  0.9430    0.9081
      Random Forest    0.9785     0.9192  0.9328    0.9260
                SVM    0.9474     0.7711  0.9027    0.8317
Logistic Regression    0.9345     0.7185  0.8963    0.7976


## 3. ADASYN

Similar to SMOTE but generates samples at the boundary between classes,

https://imbalanced-learn.org/stable/references/generated/imblearn.over_sampling.ADASYN.html

In [10]:
from imblearn.over_sampling import ADASYN

# Initialize ADASYN
adasyn = ADASYN(random_state=42)

# Apply ADASYN to the training data
X_train_adasyn, y_train_adasyn = adasyn.fit_resample(X_train, y_train)

print(f"Original training set shape: {X_train.shape}")
print(f"Distribution before ADASYN: {y_train.value_counts()}")
print('-' * 20)
print(f"ADASYN training set shape: {X_train_adasyn.shape}")
print(f"Distribution after ADASYN: {y_train_adasyn.value_counts()}")

Original training set shape: (92184, 22)
Distribution before ADASYN: label
0    78912
1    13272
Name: count, dtype: int64
--------------------
ADASYN training set shape: (157826, 22)
Distribution after ADASYN: label
1    78914
0    78912
Name: count, dtype: int64


In [11]:
df_ADASYN_class = benchmark_models(X_train_adasyn, X_test, y_train_adasyn, y_test, use_class_weights=False)
print('-' * 20)
print(df_ADASYN_class.to_string(index=False))

Training Logistic Regression...
  Done. Recall=0.9189, Features=22
Training Random Forest...
  Done. Recall=0.9394, Features=22
Training XGBoost...
  Done. Recall=0.9572, Features=22
Training LightGBM...
  Done. Recall=0.9726, Features=22
Training SVM...
  Done. Recall=0.9509, Features=22
Training Neural Network...
  Done. Recall=0.9635, Features=22
--------------------
              Model  Accuracy  Precision  Recall  F1 Score
           LightGBM    0.9380     0.7069  0.9726    0.8187
     Neural Network    0.9525     0.7665  0.9635    0.8538
            XGBoost    0.9586     0.7962  0.9572    0.8693
                SVM    0.9090     0.6200  0.9509    0.7506
      Random Forest    0.9722     0.8765  0.9394    0.9069
Logistic Regression    0.9136     0.6391  0.9189    0.7539


## 4. SMOTENC

Special variant of SMOTE that is used for datasets containing numerical and categorical features.

https://imbalanced-learn.org/stable/references/generated/imblearn.over_sampling.SMOTENC.html

In [12]:
from imblearn.over_sampling import SMOTENC

# Initialize SMOTENC
smontenc = SMOTENC(categorical_features=['is_ip', 'is_https'], random_state=42)

# Apply SMOTENC to the training data
X_train_smontenc, y_train_smontenc = smontenc.fit_resample(X_train, y_train)

print(f"Original training set shape: {X_train.shape}")
print(f"Distribution before SMOTENC: {y_train.value_counts()}")
print('-' * 20)
print(f"SMOTENC training set shape: {X_train_smontenc.shape}")
print(f"Distribution after SMOTENC: {y_train_smontenc.value_counts()}")

Original training set shape: (92184, 22)
Distribution before SMOTENC: label
0    78912
1    13272
Name: count, dtype: int64
--------------------
SMOTENC training set shape: (157824, 22)
Distribution after SMOTENC: label
0    78912
1    78912
Name: count, dtype: int64


In [13]:
df_SMOTENC_class = benchmark_models(X_train_smontenc, X_test, y_train_smontenc, y_test, use_class_weights=False)
print('-' * 20)
print(df_SMOTENC_class.to_string(index=False))

Training Logistic Regression...
  Done. Recall=0.8966, Features=22
Training Random Forest...
  Done. Recall=0.934, Features=22
Training XGBoost...
  Done. Recall=0.9397, Features=22
Training LightGBM...
  Done. Recall=0.9424, Features=22
Training SVM...
  Done. Recall=0.9027, Features=22
Training Neural Network...
  Done. Recall=0.9343, Features=22
--------------------
              Model  Accuracy  Precision  Recall  F1 Score
           LightGBM    0.9738     0.8836  0.9424    0.9121
            XGBoost    0.9785     0.9136  0.9397    0.9265
     Neural Network    0.9815     0.9368  0.9343    0.9356
      Random Forest    0.9789     0.9204  0.9340    0.9272
                SVM    0.9477     0.7725  0.9027    0.8325
Logistic Regression    0.9347     0.7189  0.8966    0.7980


## 5. Balanced Tree

Special variant of Random Forest that addresses the imbalance problem. It works by  balancing data used to train each decision tree in the forest.

https://medium.com/@fadleemt/balanced-random-forest-d5dc9c896bb4

In [11]:
from imblearn.ensemble import BalancedRandomForestClassifier

# Initialize BalancedRandomForestClassifier
brf_model = BalancedRandomForestClassifier(n_estimators=100, random_state=42, sampling_strategy='auto')

# Train the model
print('Training Balanced Random Forest...')
brf_model.fit(X_train, y_train)

# Make predictions on the test set
y_pred_brf = brf_model.predict(X_test)

# Evaluate the model
results_brf = {
    'Model': 'Balanced Random Forest',
    'Accuracy':  round(accuracy_score(y_test, y_pred_brf), 4),
    'Precision': round(precision_score(y_test, y_pred_brf, pos_label=1), 4),
    'Recall':    round(recall_score(y_test, y_pred_brf, pos_label=1), 4),
    'F1 Score':  round(f1_score(y_test, y_pred_brf, pos_label=1), 4),
}
df_brf_class = pd.DataFrame([results_brf])
print(f"  Done. Recall={results_brf['Recall']}, Features={brf_model.n_features_in_}")
print('-' * 20)
print(df_brf_class.to_string(index=False))


Training Balanced Random Forest...
  Done. Recall=0.9608, Features=22
--------------------
                 Model  Accuracy  Precision  Recall  F1 Score
Balanced Random Forest    0.9639     0.8195  0.9608    0.8846


---

# 🤖 Machine Learning 2 (Feature)

- Find out whether trimming features is possible
- No correct way to trim I think, just as long as logical
- If too much recall loss, then just keep 24 features
- Experiment with:
  - **Full set** (24 features)
  - **Correlation-based** (top 10 or certain threshold)
  - **Random Forest** Importance

---

# 📈 Machine Learning 3 (Combine)

- Evaluate best models from ML1 & ML2 (recall).
- **Hyperparameter tuning** (recall score)
- If possible, **stack/combine models** like senior report to be more accurate (if model accurate enough can ignore this)